### Phase 1 — Look at the data (~45 min)

In [3]:
from datasets import load_dataset
from pprint import pprint

In [4]:
ds = load_dataset("obalcells/longfact-annotations", "Meta-Llama-3.1-8B-Instruct", split="train")

In [7]:
ds
ds.column_names
len(ds)
ds[0]

{'subset': 'longfact_objects',
 'model': 'Meta-Llama-3.1-8B-Instruct',
 'conversation': [{'content': 'Who is William James? Provide as many specific details and examples as possible (such as names of people, numbers, events, locations, dates, times, etc.)\n',
   'role': 'user'},
  {'content': 'William James (January 11, 1842 – August 26, 1910) was an American philosopher, psychologist, and theologian. He was a leading figure in the development of pragmatism, a philosophical movement that emphasizes practicality and effectiveness in understanding and addressing complex issues.\n\n**Early Life and Education**\n\nWilliam James was born in New York City to Henry James Sr. and Mary Robertson Walsh James. His father was a theologian and social reformer, and his brothers, Henry James Jr. and William James Jr., were also notable writers. William James studied at Harvard University, where he earned his Bachelor\'s degree in 1861 and his Master\'s degree in 1863. He then traveled to Europe, wher

In [8]:
pprint(ds[0]['annotations'])                            

[{'index': 15,
  'label': 'Supported',
  'span': 'January 11, 1842',
  'verification_note': 'Multiple sources confirm William James was born on '
                       'January 11, 1842'},
 {'index': 34,
  'label': 'Supported',
  'span': 'August 26, 1910',
  'verification_note': 'Multiple sources confirm William James died on August '
                       '26, 1910'},
 {'index': 362,
  'label': 'Supported',
  'span': 'Henry James Sr.',
  'verification_note': "Confirmed that William James's father was Henry James "
                       'Sr., a theologian'},
 {'index': 382,
  'label': 'Supported',
  'span': 'Mary Robertson Walsh James',
  'verification_note': 'Sources confirm his mother was Mary Robertson Walsh '
                       '(James)'},
 {'index': 477,
  'label': 'Supported',
  'span': 'Henry James Jr.',
  'verification_note': 'Confirmed that Henry James the novelist was William '
                       "James's brother"},
 {'index': 497,
  'label': 'Not Supported',
  'sp

In [10]:
ds[0]['conversation'][1]['content'][362:380]     

'Henry James Sr. an'

In [12]:
from collections import Counter                                                                                  
Counter(a['label'] for ex in ds for a in ex['annotations'])

Counter({'Supported': 5621,
         'Not Supported': 1718,
         'Insufficient Information': 1133,
         None: 126})

In [14]:
import statistics                                         
gen_lens = [len(ex['conversation'][1]['content']) for ex in ds]
ann_counts = [len(ex['annotations']) for ex in ds]                                                               
print(f"generations: median {statistics.median(gen_lens):.0f} chars, max {max(gen_lens)}")
print(f"annotations per example: median {statistics.median(ann_counts):.0f}, max {max(ann_counts)}")             
print(f"total examples: {len(ds)}")

generations: median 4930 chars, max 10118
annotations per example: median 16, max 43
total examples: 510


In [ ]:
"""
Open question, evidently the max_length = 1536 (which is 1024+512) was set by the authors as a hyper-param, not sure whyyyy
"""

### Phase 2 — Span → token alignment (~45 min) **[TODAY]**
- Read `utils/tokenization.py`. Annotations give *character spans* in the
  generation; the probe needs *per-token labels*. This file does that
  alignment.
- Run an example through the Llama 3.1 tokenizer on CPU. Tokenizer is gated;
  request access at huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
  (instant approval), then `huggingface-cli login`. Tokenizer download is
  ~5 MB.

Read utils.tokenization.find_string_in_tokens ... its effectively doing Binary Search twice to find the beginning and the end of the span within the text, very clever

⏺ Span ↔ token alignment in utils/tokenization.py                                                                  
                                         
  Annotations come as (char_index, span_text, label) where char_index is an offset into the assistant's generation 
  string. The probe needs per-token labels in the chat-formatted full input — but those two strings differ (chat
  template adds <|begin_of_text|>, role headers, etc.), so HuggingFace's offset_mapping doesn't fit. Instead       
  find_string_in_tokens does string-search via tokenizer.decode(): two sequential binary searches, first for the   
  smallest end-index e where decode(tokens[:e]) contains the target span, then for the largest start-index s where 
  decode(tokens[s:e]) still contains it. The result slice(s, e) is the tightest token range matching the span —    
  O(log n) decode calls (~22 for a 1500-token sequence vs. 1500 for a linear scan).                                

  Concrete example: annotation {'index': 362, 'span': 'Henry James Sr.', 'label': 'Supported'}. After tokenization,
   suppose those characters land at tokens 145–149. Search 1 finds the smallest e such that decode(tokens[:e])
  contains "Henry James Sr." — that's e=150 (slice end is exclusive). Search 2 then finds the largest s such that  
  decode(tokens[s:150]) still contains it — that's s=145. Result: slice(145, 150). Those five token positions get
  the Supported (negative-class) label; the other ~1500 tokens in the sequence are loss-masked unless they fall
  within another annotation's span.

  The function alone returns the first occurrence within the tokens it's given. The caller in probe/dataset.py     
  handles duplicates by tracking a cur_idx cursor — calls find_string_in_tokens(span, input_ids[cur_idx:]) and
  advances cur_idx past each match — which works as long as annotations are sorted by char index (they are in this 
  dataset). Separately, find_assistant_tokens_slice locates the chat-template marker for the assistant turn so
  non-assistant tokens get loss-masked.

## Phase 3: HF data → training tensors (probe/dataset.py + dataset_converters.py)                                   
                                                                                                                   
The pipeline: create_probing_dataset loads from HF → dataset_converters.py normalizes the various HF formats into
   uniform ProbingItem(prompt, completion, spans) objects → TokenizedProbingDataset._process_item tokenizes (chat
  template applied, padded to max_length=1536) and assigns per-token labels.                                       
                                                                                                                   
  Label mapping (dataset_converters.py):                                                                           
                                                                                                                   
  'Not Supported'            → 1.0   (positive)                                                                    
  'Insufficient Information' → 1.0   (also positive — surprising! not masked)
  'Supported'                → 0.0   (negative)
  'N/A' / None               → -100  (ignore)                                                                      
   
  I'd previously guessed Insufficient Information would be loss-masked. It isn't — the authors deliberately count  
  "judge couldn't verify" as a hallucination. Worth flagging for Week 3: you'd decide separately what to do with
  ambiguous taxonomy labels.                                                                                       
                                                            
  Worked example. Take this row:

  prompt:     "Tell me about William James."
  generation: "William James was born in Boston on January 11, 1842."
  annotations:
    - "Boston",           "Not Supported", index=27   (positive — hallucination)                                   
    - "January 11, 1842", "Supported",     index=37   (negative — verifiable)
                                                                                                                   
  After chat template + tokenization + span alignment via find_string_in_tokens, build two parallel tensors of     
  length 1536:
                                                                                                                   
  ┌───────────┬─────────────────────────────┬───────┬───────────────────┐
  │ positions │           content           │ label │      weight       │
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤
  │ 0–12      │ prompt + chat template      │ -100  │ —                 │
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤
  │ 13–17     │ "William James was born in" │ 0.0   │ 1.0               │                                          
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤
  │ 18        │ "Boston" ← hallucination    │ 1.0   │ 10.0 (pos_weight) │                                          
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤                                          
  │ 19–20     │ " on"                       │ 0.0   │ 1.0               │
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤                                          
  │ 21–25     │ "January 11, 1842"          │ 0.0   │ 10.0 (neg_weight) │
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤                                          
  │ 26        │ "."                         │ 0.0   │ 1.0               │
  ├───────────┼─────────────────────────────┼───────┼───────────────────┤                                          
  │ 27–1535   │ template end + padding      │ -100  │ —                 │
  └───────────┴─────────────────────────────┴───────┴───────────────────┘

  Two-rule masking story:                                                                                          
   
  - label = -100 ⇒ token is skipped in the loss entirely. PyTorch's BCE-with-ignore-index handles this — those     
  positions contribute zero loss regardless of the probe's prediction.
  - weight scales the per-token loss for tokens that aren't skipped. Annotated entity tokens get                   
  pos_weight/neg_weight (10 each — heavy supervision); filler assistant tokens get 1.0 (light background           
  supervision). This is what counterbalances the rare-positive imbalance.
                                                                                                                   
  In our toy: position 18 contributes 10 * BCE(logit_18, 1.0); positions 21–25 each contribute 10 * BCE(logit_i,   
  0.0); positions 13–17, 19–20, 26 each contribute 1 * BCE(logit_i, 0.0); everything else contributes 0.
                                                                                                                   
  What ends up as -100 (loss-skipped):                                                                             
   
  1. Padding tokens.                                                                                               
  2. All tokens before the assistant turn starts (prompt + chat template).
  3. Tokens within ignore_buffer of a hallucination span — buffer applies only to positives (asymmetric on purpose:
   tokens adjacent to a hallucination are semantically ambiguous; tokens adjacent to a supported fact aren't).     
  4. Annotations explicitly labeled N/A/None.
  5. If default_ignore=True: every token not inside any annotated span. (Used for trivia_qa.)                      
  6. If default_ignore=False (longform default): non-span assistant tokens get label 0.0, weight 1.0 — they DO
  contribute to the loss as cheap negatives.                                                                       
                                                            
  That last point is easy to miss. In the longform default, the probe is trained as "hallucinated entity tokens vs.
   everything else in the assistant turn" — not just "hallucinated vs. supported entities." Most of the assistant
  turn is implicitly negative supervision at low weight. pos_weight=10 is what stops the probe from collapsing to  
  predicting all zeros.                                     

  Per-example output tuple:                                                                                        
   
  input_ids                 length-1536 token IDs                                                                  
  attention_mask            length-1536 1/0 (real vs padding)
  classification_labels     length-1536 ∈ {-100, 0, 1}    ← what to predict
  classification_weights    length-1536 floats             ← how much each token matters                           
  lm_labels                 length-1536 token IDs          (LM regularizer only; vanilla probe ignores)
  pos_spans, neg_spans      lists of [start, end] indices  (for visualization/eval)                                
                                                            
  The (classification_labels, classification_weights) pair is the loss-masking: labels=-100 is the hard filter,    
  weights are the soft scaling.                             
                                                                                                                   
  Other knobs:                                              
  - last_span_token (default False): if True, only the final token of each span gets labeled (rest of span
  ignored). Stricter signal — tested but not the default.                                                          
  - lm_labels is for the LM regularizer (only used when lambda_lm > 0); for the vanilla linear probe, ignore.
                                                                                                                   
  Bridge to Week 3. The default_ignore knob will matter a lot for sparse Sharma-taxonomy labels — if your positives
   are rare disempowerment spans within long benign conversations, the implicit "everything else is negative at    
  weight 1" supervision could dominate. May want default_ignore=True for cleaner signal early on.

  ❯ so to be clear like if theres a false positive, that still counts, but the class_weight is 1, not 10 or 100?     
                                                                                                                   
⏺ Yes, exactly right. To make it crisp:                                                                            
                                         
  - Filler token (e.g., "the", "was" — not in any annotation): label 0.0, weight 1.0. If the probe fires on it     
  (predicts 1.0), that's a false positive and contributes 1.0 × BCE(logit, 0.0) to the loss. Probe gets penalized, 
  but at 1x not 10x.                                                                                               
  - Annotated supported entity token (e.g., "January 11, 1842"): label 0.0, weight 10.0. False positive there      
  contributes 10.0 × BCE(logit, 0.0). Same kind of error, 10x the penalty.                                         
  - Annotated hallucinated entity token (e.g., "Boston" if it's wrong): label 1.0, weight 10.0. Missing it         
  (predicting 0.0) contributes 10.0 × BCE(logit, 1.0).                                                             
  - Prompt / padding tokens: label -100, skipped entirely. Probe can predict anything, no loss.
                                                                                                                   
  So the gradient signal looks like:                        
  - "When you see an annotated entity, get it really right (10x)."                                                 
  - "Everywhere else in the assistant turn, default to not firing (1x)."                                           
  - "Outside the assistant turn, do whatever — we won't grade you."     
                                                                                                                   
  The 10x weighting on annotations vs 1x on filler is what tilts the probe toward "treat annotated regions as the  
  high-value examples" while still training a global "don't fire spuriously" prior on everything else.  